In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

# ===== LOAD DATA =====
matches = pd.read_csv('../data/matches.csv')
deliveries = pd.read_csv('../data/deliveries.csv')

# ===== STANDARDIZE TEAM NAMES =====
team_name_changes = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    matches[col] = matches[col].replace(team_name_changes)
for col in ['batting_team', 'bowling_team']:
    deliveries[col] = deliveries[col].replace(team_name_changes)

# ===== 1ST INNINGS ONLY + ADD VENUE =====
df = deliveries[deliveries['inning'] == 1].copy()
df = df.merge(matches[['id', 'venue']], left_on='match_id', right_on='id')
df = df.sort_values(['match_id', 'over', 'ball']).reset_index(drop=True)

# ===== BASE FEATURES =====
df['current_runs'] = df.groupby('match_id')['total_runs'].cumsum()
df['current_wickets'] = df.groupby('match_id')['is_wicket'].cumsum()
df['ball_number'] = df.groupby('match_id').cumcount() + 1
df['overs_decimal'] = (df['ball_number'] / 6).round(2)
df['runs_last_5'] = df.groupby('match_id')['total_runs'].rolling(window=30, min_periods=1).sum().reset_index(drop=True)
df['wickets_last_5'] = df.groupby('match_id')['is_wicket'].rolling(window=30, min_periods=1).sum().reset_index(drop=True)

# ===== FINAL SCORE =====
final_scores = df.groupby('match_id')['current_runs'].max().reset_index()
final_scores.columns = ['match_id', 'final_score']
df = df.merge(final_scores, on='match_id')

# ===== FILTER CURRENT TEAMS + DROP EARLY OVERS =====
current_teams = [
    'Royal Challengers Bengaluru', 'Mumbai Indians', 'Chennai Super Kings',
    'Kolkata Knight Riders', 'Rajasthan Royals', 'Sunrisers Hyderabad',
    'Delhi Capitals', 'Punjab Kings', 'Lucknow Super Giants', 'Gujarat Titans'
]
df = df[df['batting_team'].isin(current_teams)]
df = df[df['bowling_team'].isin(current_teams)]
df = df[df['overs_decimal'] >= 5.0].copy()

# ===== SMART DERIVED FEATURES =====
df['crr'] = (df['current_runs'] / df['overs_decimal']).round(2)
df['overs_left'] = (20 - df['overs_decimal']).round(2)
df['wickets_left'] = 10 - df['current_wickets']
df['momentum'] = (df['runs_last_5'] - (df['crr'] * 5)).round(2)

# ===== SELECT FINAL FEATURES =====
final_df = df[[
    'batting_team', 'bowling_team', 'venue',
    'current_runs', 'wickets_left', 'crr', 'overs_left', 'momentum', 'wickets_last_5',
    'final_score'
]].dropna()

print(f"✅ Dataset shape: {final_df.shape}")
print(f"✅ Features: {final_df.columns.tolist()}\n")

# ===== TRAIN MODEL =====
X = final_df.drop(columns=['final_score'])
y = final_df['final_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), ['batting_team', 'bowling_team', 'venue'])],
    remainder='passthrough'
)
pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', LinearRegression())])
pipeline.fit(X_train, y_train)

# ===== EVALUATE =====
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"📊 Model Performance:")
print(f"   R² Score: {r2:.4f}")
print(f"   MAE: {mae:.2f} runs\n")

# ===== WEIGHTS =====
model = pipeline.named_steps['model']
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

print(f"📐 Bias: {model.intercept_:.4f}\n")
print("🔍 Weights for numeric features:")
print("-" * 50)
for fname, weight in zip(feature_names, model.coef_):
    if fname.startswith('remainder__'):
        clean_name = fname.replace('remainder__', '')
        print(f"   {clean_name:<20} → {weight:>10.4f}")

✅ Dataset shape: (85905, 10)
✅ Features: ['batting_team', 'bowling_team', 'venue', 'current_runs', 'wickets_left', 'crr', 'overs_left', 'momentum', 'wickets_last_5', 'final_score']

📊 Model Performance:
   R² Score: 0.6835
   MAE: 13.45 runs

📐 Bias: 9.4295

🔍 Weights for numeric features:
--------------------------------------------------
   current_runs         →     0.7056
   wickets_left         →     4.4521
   crr                  →     3.8071
   overs_left           →     4.4351
   momentum             →    -0.1349
   wickets_last_5       →    -1.1235


In [2]:
sample = pd.DataFrame({
    'batting_team': ['Chennai Super Kings'],
    'bowling_team': ['Mumbai Indians'],
    'venue': ['Wankhede Stadium'],
    'current_runs': [120],
    'wickets_left': [7],          # 3 wickets fallen
    'crr': [8.57],                # 120 runs / 14 overs
    'overs_left': [6.0],
    'momentum': [-2.85],          # 55 in last 5 (12 expected based on crr)
    'wickets_last_5': [1]
})

predicted = pipeline.predict(sample)[0]
print(f"🏏 CSK 120/3 in 14 overs at Wankhede")
print(f"📊 Predicted final score: {predicted:.0f}")

🏏 CSK 120/3 in 14 overs at Wankhede
📊 Predicted final score: 187


In [3]:
import pickle
import os

# Create models folder if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the entire pipeline (preprocessor + model)
with open('../models/score_predictor.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

# Also save the list of teams and venues for dropdowns in UI
metadata = {
    'teams': current_teams,
    'venues': sorted(final_df['venue'].unique().tolist())
}

with open('../models/metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("✅ Model saved to ../models/score_predictor.pkl")
print("✅ Metadata saved to ../models/metadata.pkl")
print(f"📁 Total venues saved: {len(metadata['venues'])}")

✅ Model saved to ../models/score_predictor.pkl
✅ Metadata saved to ../models/metadata.pkl
📁 Total venues saved: 54
